# Context-Scoped SyncManager

## What this notebook proves
- `freshness("stale_ok")` uses the default manager's route outside a context block.
- `using_sync_manager(sync_b)` overrides routing for only the block scope.
- Exiting the block restores the original default manager.

## Deterministic expectation in this demo
- We seed **2 local-only rows** into `sync_a.local`.
- We seed **1 remote-only row** into `sync_b.remote`.
- Query outside context should return `2` rows.
- Query inside `using_sync_manager(sync_b)` should return `1` row.


In [ ]:
from surrealengine import (
    Document,
    StringField,
    create_connection,
    create_sync_manager,
    SyncConfig,
    SyncPolicy,
    using_sync_manager,
)

class Ticket(Document):
    title = StringField()

    class Meta:
        collection = "tickets_ctx_sync"


In [ ]:
import os

SURREAL_URL = os.getenv("SURREAL_URL", "ws://localhost:8080/rpc")
SURREAL_USER = os.getenv("SURREAL_USER", "root")
SURREAL_PASS = os.getenv("SURREAL_PASS", "secret")

remote_a = create_connection(
    url=SURREAL_URL,
    namespace="demo",
    database="ctx_a",
    username=SURREAL_USER,
    password=SURREAL_PASS,
)
local_a = create_connection(url="mem://tmp/ctx_a", namespace="demo", database="ctx_a")
await remote_a.connect()
await local_a.connect()

remote_b = create_connection(
    url=SURREAL_URL,
    namespace="demo",
    database="ctx_b",
    username=SURREAL_USER,
    password=SURREAL_PASS,
)
local_b = create_connection(url="mem://tmp/ctx_b", namespace="demo", database="ctx_b")
await remote_b.connect()
await local_b.connect()

sync_a = create_sync_manager(
    remote=remote_a,
    local=local_a,
    config=SyncConfig(auto_max_lag_ms=500),
    make_default=True,
)
sync_a.register_model(Ticket, policy=SyncPolicy.mirrored)

sync_b = create_sync_manager(
    remote=remote_b,
    local=local_b,
    config=SyncConfig(auto_max_lag_ms=500),
    make_default=False,
)
sync_b.register_model(Ticket, policy=SyncPolicy.ignored)


In [ ]:
# Seed deterministic data into each route target.
# sync_a + stale_ok -> local_a
# sync_b + stale_ok (policy ignored) -> remote_b

# reset local_a target table
await local_a.client.query("DEFINE TABLE tickets_ctx_sync SCHEMALESS;")
await local_a.client.query("DELETE tickets_ctx_sync;")
await Ticket(id="tickets_ctx_sync:local_1", title="local-only-a").save(connection=local_a)
await Ticket(id="tickets_ctx_sync:local_2", title="local-only-b").save(connection=local_a)

# reset remote_b target table
await remote_b.client.query("DEFINE TABLE tickets_ctx_sync SCHEMALESS;")
await remote_b.client.query("DELETE tickets_ctx_sync;")
await Ticket(id="tickets_ctx_sync:remote_1", title="remote-only-b").save(connection=remote_b)

# Also clear opposite stores to keep the assertion obvious
await remote_a.client.query("DEFINE TABLE tickets_ctx_sync SCHEMALESS;")
await remote_a.client.query("DELETE tickets_ctx_sync;")
await local_b.client.query("DEFINE TABLE tickets_ctx_sync SCHEMALESS;")
await local_b.client.query("DELETE tickets_ctx_sync;")

# Default manager (sync_a) should route stale_ok -> local_a (2 rows)
rows_default = await Ticket.objects.freshness("stale_ok").all()
print("default manager rows:", len(rows_default), [r.title for r in rows_default])

# Context override (sync_b) should route stale_ok -> remote_b (1 row)
with using_sync_manager(sync_b):
    rows_ctx = await Ticket.objects.freshness("stale_ok").all()
    print("context manager rows:", len(rows_ctx), [r.title for r in rows_ctx])


In [ ]:
from surrealengine.context import get_active_sync_manager

active_before = get_active_sync_manager()
print("Default manager active:", active_before is sync_a)

with using_sync_manager(sync_b):
    active_inside = get_active_sync_manager()
    print("Scoped manager active:", active_inside is sync_b)

active_after = get_active_sync_manager()
print("Default manager restored:", active_after is sync_a)


In [ ]:
checks = {
    "default_rows_count_is_2": len(rows_default) == 2,
    "context_rows_count_is_1": len(rows_ctx) == 1,
    "default_manager_was_active": active_before is sync_a,
    "scoped_manager_was_active": active_inside is sync_b,
    "default_manager_restored": active_after is sync_a,
}
for name, ok in checks.items():
    print(f"{name}: {'PASS' if ok else 'FAIL'}")

if not all(checks.values()):
    raise AssertionError("Context-scoped deterministic checks failed")

print("ALL CONTEXT-SCOPED CHECKS PASSED")


In [ ]:
await remote_a.disconnect()
await local_a.disconnect()
await remote_b.disconnect()
await local_b.disconnect()
